# EEG Feature Extraction - Hierarchical Strategies

This notebook extracts EEG features from preprocessed EEG data using a **principled granularity hierarchy**.

**Parameterized**: Set `STRATEGY` to choose feature extraction level.

**Run this notebook ONCE per strategy after EEG preprocessing completes.**

All other model notebooks will load the saved features instead of re-extracting.

---

## Feature Extraction Hierarchy

| Level | Strategy | Features | Description | Builds On |
|-------|----------|----------|-------------|----------|
| 0 | `channels_raw` | 20 | Total power per electrode | Base |
| 1 | `regional_raw` | 4 | Regional averages of total power | channels_raw |
| 2 | `channels_bands` | 80 | 20 channels × 4 frequency bands | channels_raw |
| 3 | `regional_bands` | 16 | 4 regions × 4 frequency bands | channels_bands |
| 4 | `extended` | 160 | channels_bands + lateralization + temporal | channels_bands |

Each level builds on the previous, allowing systematic ablation studies:
- Does band decomposition help? Compare Level 0 vs Level 2
- Does regional aggregation help? Compare Level 0 vs Level 1, or Level 2 vs Level 3
- Do derived features help? Compare Level 2 vs Level 4

In [46]:
# ============================================================================
# CONFIGURATION: Set extraction strategy
# ============================================================================
STRATEGY = 'extended'  # Options: 'channels_raw', 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
# ============================================================================

import sys
sys.path.append('../..')

import pickle
import pandas as pd
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Force reload of the module to pick up any changes
import importlib
import src.features.eeg_features
importlib.reload(src.features.eeg_features)

from src.features.eeg_features import (
    extract_eeg_features,
    get_feature_metadata,
    get_strategy_hierarchy,
    CHANNEL_NAMES,
    CHANNEL_REGIONS,
    FREQ_BANDS,
    PREPROCESSED_EEG_PKL,
)

# Display hierarchy
hierarchy = get_strategy_hierarchy()
print(f"\n{'='*70}")
print(f"EEG FEATURE EXTRACTION: {STRATEGY.upper()} (Level {hierarchy[STRATEGY]['level']})")
print(f"{'='*70}\n")
print(f"Feature extraction started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nStrategy Hierarchy:")
for strat, info in hierarchy.items():
    marker = "-->" if strat == STRATEGY else "   "
    print(f"  {marker} Level {info['level']}: {strat:20s} ({info['n_features']:3d} features)")


EEG FEATURE EXTRACTION: EXTENDED (Level 4)

Feature extraction started: 2026-04-02 14:40:19

Strategy Hierarchy:
      Level 0: channels_raw         ( 20 features)
      Level 1: regional_raw         (  4 features)
      Level 2: channels_bands       ( 80 features)
      Level 3: regional_bands       ( 16 features)
  --> Level 4: extended             (112 features)


## 1. Load Preprocessed EEG Data

Load the preprocessed EEG pickle file containing display_eeg data.

In [47]:
eeg_data_path = PREPROCESSED_EEG_PKL

print(f"Loading EEG data from: {eeg_data_path}")
with open(eeg_data_path, 'rb') as f:
    eeg_df = pickle.load(f)

print(f"\n✓ Loaded {len(eeg_df)} trials")
print(f"  Unique subjects: {eeg_df['subject_date_id'].nunique()}")

# Check EEG data structure
sample_eeg = eeg_df['review_eeg'].iloc[0]
print(f"  EEG array shape: {sample_eeg.shape} (time × channels)")
print(f"\nColumns: {eeg_df.columns.tolist()}")

Loading EEG data from: /Users/pranmodu/Projects/columbia/liinc_eye/data/eeg/preprocessed_eeg_review.pkl

✓ Loaded 11806 trials
  Unique subjects: 94
  EEG array shape: (20, 512) (time × channels)

Columns: ['subject_date_id', 'trial_id', 'review_eeg', 'baseline_eeg']


## 2. Channel and Region Configuration

Display the channel configuration used for feature extraction.

In [48]:
print("\n" + "="*70)
print("CHANNEL CONFIGURATION (from chan_locs.sfp)")
print("="*70)

print(f"\n20 EEG Channels (10-20 system):")
print(f"  {', '.join(CHANNEL_NAMES)}")

print(f"\n4 Brain Regions:")
for region, channels in CHANNEL_REGIONS.items():
    print(f"  {region:10s}: {', '.join(channels)}")

print(f"\n4 Frequency Bands:")
for band, (f_low, f_high) in FREQ_BANDS.items():
    print(f"  {band:6s}: {f_low:4.1f} - {f_high:4.1f} Hz")


CHANNEL CONFIGURATION (from chan_locs.sfp)

20 EEG Channels (10-20 system):
  Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2

4 Brain Regions:
  Frontal   : Fp1, Fp2, F7, F3, Fz, F4, F8
  Central   : T3, C3, Cz, C4, T4
  Parietal  : T5, P3, Pz, P4, T6
  Occipital : O1, POz, O2

4 Frequency Bands:
  Delta :  0.5 -  4.0 Hz
  Theta :  4.0 -  8.0 Hz
  Alpha :  8.0 - 13.0 Hz
  Beta  : 13.0 - 30.0 Hz


## 3. Extract EEG Features

Extract features based on selected strategy:

### Level 0: channels_raw (20 features)
- Total (broadband) power per electrode
- **Features:** `eeg_{channel}` for each of 20 channels
- **Use case:** Simplest baseline, tests if spatial pattern alone carries signal

### Level 1: regional_raw (4 features)
- Regional averages of total power
- **Features:** `eeg_{Frontal,Central,Parietal,Occipital}`
- **Use case:** Most compact representation, tests regional differences

### Level 2: channels_bands (80 features)
- Band power per channel (20 channels × 4 bands)
- **Features:** `eeg_{band}_{channel}`
- **Use case:** Full spatial + spectral resolution

### Level 3: regional_bands (16 features)
- Regional average band power (4 regions × 4 bands)
- **Features:** `eeg_{band}_{region}`
- **Use case:** Standard approach, balances detail with interpretability

### Level 4: extended (160 features)
- channels_bands + temporal dynamics + lateralization
- **Channel-band power:** 80 features
- **Temporal dynamics:** `eeg_{band}_{region}_{mean,std,slope}` (48 features)
- **Lateralization:** `eeg_{band}_{pair}_lateralization` (32 features)
- **Use case:** Maximum feature richness for best performance

In [49]:
print(f"\nExtracting EEG features using '{STRATEGY}' strategy...\n")

eeg_features_df = extract_eeg_features(
    eeg_df=eeg_df,
    strategy=STRATEGY,
    fs=256,
    verbose=True
)

# Get feature columns
eeg_cols = [c for c in eeg_features_df.columns if c.startswith('eeg_')]

print(f"\n{'='*70}")
print(f"✓ Extracted {len(eeg_cols)} EEG features")
print(f"  Example features: {eeg_cols[:5]}")
if len(eeg_cols) > 5:
    print(f"  ... and {len(eeg_cols) - 5} more")
print(f"{'='*70}")


Extracting EEG features using 'extended' strategy...

Extracting EEG features using 'extended' strategy (Level 4)...
  EEG column: review_eeg
  Sampling rate: 256 Hz
  Trials: 11806
✓ Extracted 160 EEG features
  Channel-band power: 80 features
  Temporal dynamics: 48 features
  Lateralization: 32 features

✓ Extracted 160 EEG features
  Example features: ['eeg_Delta_Fp1', 'eeg_Delta_F7', 'eeg_Delta_F8', 'eeg_Delta_T4', 'eeg_Delta_T6']
  ... and 155 more


## 4. Inspect Features

In [50]:
# Display sample data
print("\nSample EEG features:")
display(eeg_features_df.head(3))

# Feature statistics
print("\nFeature statistics:")
display(eeg_features_df[eeg_cols].describe())


Sample EEG features:


,subject_id,trial_id,eeg_Delta_Fp1,eeg_Delta_F7,eeg_Delta_F8,eeg_Delta_T4,eeg_Delta_T6,eeg_Delta_T5,eeg_Delta_T3,eeg_Delta_Fp2,...,eeg_Beta_Frontal_slope,eeg_Beta_Central_mean,eeg_Beta_Central_std,eeg_Beta_Central_slope,eeg_Beta_Parietal_mean,eeg_Beta_Parietal_std,eeg_Beta_Parietal_slope,eeg_Beta_Occipital_mean,eeg_Beta_Occipital_std,eeg_Beta_Occipital_slope
0,0831_1300_9M4VCHG,0_0831_1300_9M4VCHG,0.008724,0.008277,0.003570,0.006190,0.004129,0.009391,0.006230,0.010668,...,0.000163,0.001707,0.001593,0.000471,0.000947,0.000837,0.000217,0.000503,0.000415,0.000121
1,0831_1300_9M4VCHG,1_0831_1300_9M4VCHG,0.000274,0.000371,0.001344,0.004270,0.000776,0.001576,0.001787,0.001250,...,-0.000001,0.000674,0.000143,-0.000005,0.000381,0.000136,-0.000014,0.000321,0.000155,-0.000009
2,0831_1300_9M4VCHG,2_0831_1300_9M4VCHG,0.000110,0.000441,0.001263,0.000781,0.001025,0.000611,0.000782,0.001372,...,0.000061,0.000662,0.000278,0.000076,0.000351,0.000155,0.000070,0.000359,0.000170,0.000069



Feature statistics:


,eeg_Delta_Fp1,eeg_Delta_F7,eeg_Delta_F8,eeg_Delta_T4,eeg_Delta_T6,eeg_Delta_T5,eeg_Delta_T3,eeg_Delta_Fp2,eeg_Delta_O1,eeg_Delta_P3,...,eeg_Beta_Frontal_slope,eeg_Beta_Central_mean,eeg_Beta_Central_std,eeg_Beta_Central_slope,eeg_Beta_Parietal_mean,eeg_Beta_Parietal_std,eeg_Beta_Parietal_slope,eeg_Beta_Occipital_mean,eeg_Beta_Occipital_std,eeg_Beta_Occipital_slope
count,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,...,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000,11806.000000
mean,0.102770,0.096413,0.100315,0.088008,0.092058,0.107004,0.106196,0.115049,0.099035,0.089432,...,0.001642,0.048378,0.028517,0.001579,0.043977,0.022045,0.001187,0.039639,0.021723,0.001769
std,0.230416,0.287857,0.281664,0.266410,0.194657,0.205775,0.275275,0.324309,0.227727,0.220125,...,0.035590,0.195410,0.213210,0.072271,0.104158,0.094097,0.033092,0.090495,0.090644,0.030234
min,0.000029,0.000044,0.000035,0.000041,0.000032,0.000035,0.000036,0.000078,0.000015,0.000008,...,-1.323128,0.000019,0.000003,-3.662856,0.000017,0.000002,-1.880892,0.000017,0.000003,-0.502376
25%,0.002298,0.002368,0.002317,0.001887,0.001664,0.001502,0.001742,0.002726,0.001482,0.001043,...,-0.000119,0.000604,0.000188,-0.000110,0.000490,0.000168,-0.000094,0.000466,0.000174,-0.000135
50%,0.017464,0.013776,0.013859,0.010762,0.011976,0.010905,0.012713,0.018365,0.013601,0.009416,...,0.000022,0.003999,0.001680,0.000012,0.004158,0.001636,0.000013,0.004282,0.001713,0.000009
75%,0.132117,0.124392,0.128126,0.108317,0.128692,0.163854,0.147210,0.132203,0.130995,0.116944,...,0.000940,0.058496,0.018792,0.000689,0.060964,0.019692,0.000688,0.060130,0.023140,0.000745
max,6.261061,21.633033,15.069611,17.314182,6.593146,6.070624,18.822366,6.186368,8.616787,8.509874,...,1.837038,11.721966,9.736610,3.080007,3.477843,4.042921,1.202239,5.315096,7.172628,2.295852


## 5. Categorize Features by Type

Group features by their type based on the extraction strategy.

In [51]:
feature_categories = {}

if STRATEGY == 'channels_raw':
    feature_categories['channel_power'] = eeg_cols
    print(f"\nChannel power features: {len(feature_categories['channel_power'])}")
    print(f"  Channels: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'regional_raw':
    feature_categories['regional_power'] = eeg_cols
    print(f"\nRegional power features: {len(feature_categories['regional_power'])}")
    print(f"  Regions: {[c.replace('eeg_', '') for c in eeg_cols]}")

elif STRATEGY == 'channels_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_channels'] = band_cols
    print(f"\nChannel-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'regional_bands':
    for band in FREQ_BANDS.keys():
        band_cols = [c for c in eeg_cols if f'eeg_{band}_' in c]
        feature_categories[f'{band}_regional'] = band_cols
    print(f"\nRegional-band power features by band:")
    for band, cols in feature_categories.items():
        print(f"  {band}: {len(cols)} features")

elif STRATEGY == 'extended':
    # Separate by feature type
    channel_band_cols = [c for c in eeg_cols if not any(x in c for x in 
                        ['_mean', '_std', '_slope', '_lateralization'])]
    temporal_cols = [c for c in eeg_cols if any(x in c for x in 
                    ['_mean', '_std', '_slope'])]
    lat_cols = [c for c in eeg_cols if '_lateralization' in c]
    
    feature_categories['channel_band_power'] = channel_band_cols
    feature_categories['temporal_dynamics'] = temporal_cols
    feature_categories['lateralization'] = lat_cols
    
    print(f"\nExtended features by type:")
    print(f"  Channel-band power: {len(channel_band_cols)} features")
    print(f"  Temporal dynamics: {len(temporal_cols)} features")
    print(f"  Lateralization: {len(lat_cols)} features")

print(f"\nTotal features: {len(eeg_cols)}")


Extended features by type:
  Channel-band power: 80 features
  Temporal dynamics: 48 features
  Lateralization: 32 features

Total features: 160


## 6. Prepare Metadata

Create comprehensive metadata for reproducibility.

In [52]:
metadata = get_feature_metadata(STRATEGY)
metadata.update({
    'extraction_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'n_trials': len(eeg_features_df),
    'n_subjects': eeg_features_df['subject_id'].nunique(),
    'input_file': eeg_data_path,
    'description': f'EEG features extracted using {STRATEGY} strategy (Level {metadata["level"]})'
})

print("\nMetadata:")
for key, value in metadata.items():
    if not isinstance(value, (dict, list)):
        print(f"  {key}: {value}")


Metadata:
  strategy: extended
  level: 4
  builds_on: channels_bands
  sampling_rate: 256
  n_channels: 20
  n_regions: 4
  n_features: 160
  extraction_date: 2026-04-02 14:41:53
  n_trials: 11806
  n_subjects: 94
  input_file: /Users/pranmodu/Projects/columbia/liinc_eye/data/eeg/preprocessed_eeg_review.pkl
  description: EEG features extracted using extended strategy (Level 4)


## 7. Save to Pickle File

Save features with metadata for use in other notebooks.

In [53]:
# Output path based on strategy
output_dir = Path('../../data/features')
output_dir.mkdir(parents=True, exist_ok=True)

# Strategy-to-filename mapping
strategy_filenames = {
    'channels_raw': 'eeg_features_channels_raw.pkl',
    'regional_raw': 'eeg_features_regional_raw.pkl',
    'channels_bands': 'eeg_features_channels_bands.pkl',
    'regional_bands': 'eeg_features_regional_bands.pkl',
    'extended': 'eeg_features_extended.pkl'
}

output_filename = strategy_filenames[STRATEGY]
output_path = output_dir / output_filename

# Prepare output data (matching main feature extraction format)
output_data = {
    'eeg_features_df': eeg_features_df,
    'feature_columns': eeg_cols,
    'feature_categories': feature_categories,
    'metadata': metadata
}

# Save
with open(output_path, 'wb') as f:
    pickle.dump(output_data, f)

file_size_mb = output_path.stat().st_size / 1024 / 1024

print(f"\n{'='*70}")
print(f"✓ EEG features saved to: {output_path}")
print(f"  File size: {file_size_mb:.2f} MB")
print(f"  Trials: {len(eeg_features_df)}")
print(f"  Subjects: {eeg_features_df['subject_id'].nunique()}")
print(f"  Features: {len(eeg_cols)}")
print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*70}")


✓ EEG features saved to: ../../data/features/eeg_features_extended.pkl
  File size: 14.70 MB
  Trials: 11806
  Subjects: 94
  Features: 160

Completed: 2026-04-02 14:41:53


## 8. Verification

Verify the saved file can be loaded correctly.

In [54]:
# Test loading
print("\nVerifying saved file...")
with open(output_path, 'rb') as f:
    test_data = pickle.load(f)

print(f"✓ File loads successfully")
print(f"  Keys: {list(test_data.keys())}")
print(f"  Features: {len(test_data['feature_columns'])}")
print(f"  Trials: {len(test_data['eeg_features_df'])}")
print(f"  Strategy: {test_data['metadata']['strategy']}")
print(f"  Level: {test_data['metadata']['level']}")
print("\n✓ Verification complete!")


Verifying saved file...
✓ File loads successfully
  Keys: ['eeg_features_df', 'feature_columns', 'feature_categories', 'metadata']
  Features: 160
  Trials: 11806
  Strategy: extended
  Level: 4

✓ Verification complete!


---

## Usage in Other Notebooks

To use these EEG features in fusion models or other analyses:

```python
import pickle

# Load EEG features (choose the strategy you want)
strategy = 'channels_raw'  # or 'regional_raw', 'channels_bands', 'regional_bands', 'extended'
with open(f'../../data/features/eeg_features_{strategy}.pkl', 'rb') as f:
    eeg_data = pickle.load(f)

eeg_features_df = eeg_data['eeg_features_df']
eeg_cols = eeg_data['feature_columns']
metadata = eeg_data['metadata']

print(f"Loaded {metadata['strategy']} (Level {metadata['level']}): {len(eeg_cols)} features")

# Merge with other modalities
merged_df = merged_df.merge(
    eeg_features_df,
    on=['subject_id', 'trial_id'],
    how='inner'
)
```

---

## Strategy Hierarchy Reference

| Level | Strategy | Features | Builds On | Output File |
|-------|----------|----------|-----------|-------------|
| 0 | `channels_raw` | 20 | Base | `eeg_features_channels_raw.pkl` |
| 1 | `regional_raw` | 4 | channels_raw | `eeg_features_regional_raw.pkl` |
| 2 | `channels_bands` | 80 | channels_raw | `eeg_features_channels_bands.pkl` |
| 3 | `regional_bands` | 16 | channels_bands | `eeg_features_regional_bands.pkl` |
| 4 | `extended` | 160 | channels_bands | `eeg_features_extended.pkl` |

---

## Channel Configuration

All features use standardized channel configuration from `data/eeg/chan_locs.sfp`:

**20 EEG Channels (10-20 system):**
```
Fp1, F7, F8, T4, T6, T5, T3, Fp2, O1, P3, Pz, F3, Fz, F4, C4, P4, POz, C3, Cz, O2
```

**4 Brain Regions:**
- **Frontal:** Fp1, Fp2, F7, F3, Fz, F4, F8
- **Central:** T3, C3, Cz, C4, T4
- **Parietal:** T5, P3, Pz, P4, T6
- **Occipital:** O1, POz, O2

**4 Frequency Bands:**
- Delta: 0.5-4 Hz
- Theta: 4-8 Hz
- Alpha: 8-13 Hz
- Beta: 13-30 Hz